# Phase 1 — Mathematical parity showcase

This notebook is an end-user tour of Mlektic's rigorous mathematical views for linear and logistic regression. Every figure is produced by the public API from a fitted Scikit-learn estimator. There are no test assertions in this notebook.

The default `detail="essential"` keeps the compact main visualization and full metadata contract. The examples below opt into `academic` or `complete` detail to expose a fitted-model derivation, objective conventions, feature-space semantics, and conservative regularization statements. Across all detail levels, the one-feature linear view places its evolving fitted equation in a dedicated LaTeX band above the axes; the animation remains fluid and retains its replay/interpolation provenance.

In [ ]:
import numpy as np
from IPython.display import display
from sklearn.linear_model import (
    LinearRegression,
    LogisticRegression,
    SGDClassifier,
    SGDRegressor,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from mlektic import visualize_logistic, visualize_lr

rng = np.random.default_rng(42)

## 1. Fluid one-feature linear replay

`steps=30` constructs 30 semantic states: 29 reconstructed replay states followed by the exact supplied estimator, explicitly labeled `fitted`. `max_frames=10` retains 10 states for the slider and always retains that endpoint. In automatic mode, one-feature linear regression uses hybrid trace interpolation; `interpolation_frames=4` inserts perceptual subframes without pretending they are extra optimizer steps. The evolving LaTeX equation therefore reaches the same final coefficients shown in the separate academic panel, without requiring per-frame layout redraws.

In [ ]:
X_linear_1d = np.linspace(-3.0, 3.0, 100).reshape(-1, 1)
y_linear_1d = 1.4 + 2.1 * X_linear_1d[:, 0] + rng.normal(0.0, 0.35, 100)
linear_sgd = SGDRegressor(max_iter=3000, tol=1e-4, random_state=7).fit(X_linear_1d, y_linear_1d)

linear_replay = visualize_lr(
    linear_sgd,
    X_linear_1d,
    y_linear_1d,
    steps=30,
    max_frames=10,
    smooth="ema",
    animation_mode="auto",
    interpolation_frames=4,
    fps=30,
    detail="academic",
    feature_names=["study_hours"],
    sample_index=70,
)
display(linear_replay)

## 2. Two-feature linear model with the complete contract

A closed-form estimator cannot reveal its original training trajectory, so this figure is labeled synthetic interpolation. Complete detail adds the exact empirical MSE convention, MAE, R², feature contributions, and the explicit statement that canonical gradient equations are references rather than recovered optimizer behavior.

In [ ]:
X_linear_2d = rng.normal(size=(120, 2))
y_linear_2d = (
    0.8
    + 1.7 * X_linear_2d[:, 0]
    - 0.9 * X_linear_2d[:, 1]
    + rng.normal(0.0, 0.2, 120)
)
linear_2d_model = LinearRegression().fit(X_linear_2d, y_linear_2d)

linear_2d = visualize_lr(
    linear_2d_model,
    X_linear_2d,
    y_linear_2d,
    steps=18,
    max_frames=9,
    detail="complete",
    feature_names=["study_hours", "sleep_hours"],
    sample_index=8,
)
display(linear_2d)

## 3. High-dimensional linear contributions

The compact panel distinguishes coefficient magnitude from the selected observation's contribution $\theta_j x_j$. Coefficient-value products wrap across LaTeX rows, so all six contributions remain visible without crossing the canvas boundary. Models above nine variables display the nine largest absolute contributions with an explicit selection note; every contribution remains available in `layout.meta["mlektic_math"]`. Because `show_loss=True` is the linear default, the right panel shows `Interpolation MSE`: empirical MSE evaluated at each synthetic interpolation state, not optimizer training loss. It reaches the exact fitted-model MSE.

In [ ]:
feature_names_nd = ["attendance", "assignments", "projects", "sleep", "practice", "prior_score"]
X_linear_nd = rng.normal(size=(110, len(feature_names_nd)))
theta_nd = np.array([0.7, 1.4, 1.1, 0.2, 0.9, 1.8])
y_linear_nd = 0.3 + X_linear_nd @ theta_nd + rng.normal(0.0, 0.25, 110)
linear_nd_model = LinearRegression().fit(X_linear_nd, y_linear_nd)

linear_nd = visualize_lr(
    linear_nd_model,
    X_linear_nd,
    y_linear_nd,
    steps=12,
    max_frames=6,
    detail="complete",
    feature_names=feature_names_nd,
    sample_index=14,
)
display(linear_nd)

## 4. Honest pipeline mathematics

The first figure verifies an affine scaling map and converts coefficients back to original units. The second uses polynomial expansion, which is not affine in the original input. Mlektic therefore shows transformed-feature mathematics and does not claim that a raw-space linear coefficient vector exists.

In [ ]:
scaled_linear_model = make_pipeline(StandardScaler(), LinearRegression()).fit(
    X_linear_2d, y_linear_2d
)
scaled_linear = visualize_lr(
    scaled_linear_model,
    X_linear_2d,
    y_linear_2d,
    steps=12,
    max_frames=6,
    detail="complete",
    display_space="original",
    feature_names=["study_hours", "sleep_hours"],
    sample_index=8,
)
display(scaled_linear)

X_curve = np.linspace(-2.5, 2.5, 90).reshape(-1, 1)
y_curve = 1.0 - 0.7 * X_curve[:, 0] + 1.6 * X_curve[:, 0] ** 2
polynomial_model = make_pipeline(
    PolynomialFeatures(2, include_bias=False),
    LinearRegression(),
).fit(X_curve, y_curve)
polynomial_linear = visualize_lr(
    polynomial_model,
    X_curve,
    y_curve,
    steps=12,
    max_frames=6,
    detail="complete",
    feature_names=["input"],
    sample_index=20,
)
display(polynomial_linear)

## 5. Binary logistic chain and a custom threshold

The figure connects score, sigmoid, positive-class probability, threshold, and winning class index. Domain labels stay hidden by default to avoid unexplained words in the mathematics; fitted label order remains in metadata. The dashed threshold and its intersection with the sigmoid make the decision boundary visible.

In [ ]:
X_binary_1d = np.linspace(-3.5, 3.5, 120).reshape(-1, 1)
binary_signal = X_binary_1d[:, 0] + rng.normal(0.0, 0.65, 120)
y_binary_labels = np.where(binary_signal >= 0.0, "admitted", "not_admitted")
binary_1d_model = LogisticRegression(max_iter=1000).fit(X_binary_1d, y_binary_labels)

binary_1d = visualize_logistic(
    binary_1d_model,
    X_binary_1d,
    y_binary_labels,
    steps=24,
    max_frames=10,
    detail="academic",
    threshold=0.65,
    feature_names=["readiness_score"],
    sample_index=82,
    show_class_labels=False,
)
display(binary_1d)

## 6. Binary probability surface and decision contour

For two features, the model is a probability surface. In academic detail, a white contour marks the configured probability threshold. This is distinct from the score plane: with the default threshold 0.5, the same boundary is $z(\mathbf{x})=0$.

In [ ]:
X_binary_2d = rng.normal(size=(140, 2))
binary_score_2d = 0.25 + 1.2 * X_binary_2d[:, 0] - 0.8 * X_binary_2d[:, 1]
y_binary_2d = (binary_score_2d + rng.normal(0.0, 0.45, 140) > 0.0).astype(int)
binary_2d_model = LogisticRegression(max_iter=1000).fit(X_binary_2d, y_binary_2d)

binary_2d = visualize_logistic(
    binary_2d_model,
    X_binary_2d,
    y_binary_2d,
    steps=18,
    max_frames=9,
    detail="complete",
    threshold=0.5,
    feature_names=["evidence_1", "evidence_2"],
    sample_index=18,
)
display(binary_2d)

## 7. Multiclass Softmax with one focused probability surface

Showing every translucent class surface at once can obscure the model. `class_focus` keeps one surface visible and reports `1/K` in the title. The full score matrix, class order, probabilities, and argmax decision remain in the mathematical contract.

In [ ]:
X_multiclass = rng.normal(size=(180, 2))
class_scores = np.column_stack([
    1.2 * X_multiclass[:, 0],
    1.1 * X_multiclass[:, 1],
    -0.9 * X_multiclass[:, 0] - 0.8 * X_multiclass[:, 1],
])
class_names = np.array(["blue", "green", "red"])
y_multiclass = class_names[np.argmax(class_scores, axis=1)]
y_multiclass[:3] = class_names
multiclass_model = LogisticRegression(max_iter=1000).fit(X_multiclass, y_multiclass)

multiclass_focus = visualize_logistic(
    multiclass_model,
    X_multiclass,
    y_multiclass,
    steps=16,
    max_frames=8,
    detail="academic",
    class_focus="green",
    feature_names=["signal_1", "signal_2"],
    sample_index=25,
)
display(multiclass_focus)

## 8. Replayed multiclass OvR semantics

`SGDClassifier(loss="log_loss")` exposes incremental replay and typically resolves to normalized one-vs-rest sigmoids. The figure names that link instead of assuming Softmax, and complete detail reports the public regularization settings without inventing private scaling rules.

In [ ]:
ovr_model = SGDClassifier(
    loss="log_loss",
    penalty="elasticnet",
    alpha=0.001,
    l1_ratio=0.2,
    max_iter=3000,
    random_state=11,
).fit(X_multiclass, y_multiclass)

ovr_replay = visualize_logistic(
    ovr_model,
    X_multiclass,
    y_multiclass,
    steps=20,
    max_frames=8,
    smooth="ema",
    show_loss=True,
    detail="complete",
    class_focus=0,
    feature_names=["signal_1", "signal_2"],
    sample_index=30,
)
display(ovr_replay)

## 9. Inspecting the transparent contract

The visible panel is compact. The complete source of truth is the metadata attached to the figure. These cells display ordinary dictionaries for exploration; they are not unit tests.

In [ ]:
linear_contract = linear_2d.layout.meta["mlektic_math"]
display({
    "equation_space": linear_contract["equation_space"],
    "feature_names": linear_contract["feature_names"],
    "sample": linear_contract["sample"],
    "objective": linear_contract["objective"],
    "regularization": linear_contract["regularization"],
})

logistic_contract = multiclass_focus.layout.meta["mlektic_math"]
display({
    "classes": logistic_contract["classes"],
    "probability_link": logistic_contract["probability_link"],
    "class_focus_index": logistic_contract["class_focus_index"],
    "sample": logistic_contract["sample"],
    "objective": logistic_contract["objective"],
})

## Reading the figures responsibly

- **Replay** means Mlektic fitted a clone incrementally; it is not a recording of the original `fit()` call.
- **Synthetic interpolation** is a pedagogical path from a documented baseline to the fitted model; it is not optimizer history.
- For coefficient-bearing logistic models, each interpolated probability now comes from the coefficient state shown at that same checkpoint.
- EMA changes only displayed replay loss. Synthetic interpolation already defines a smooth path, so it displays raw empirical evaluation and reaches the exact fitted endpoint.
- The fitted-model panel is deliberately stable during playback to preserve fluid animation.
- Regularization families and public settings are shown when available; solver-private normalization and intercept treatment are not guessed.